# Slow Control Data Access and P-T Phase Diagram

This notebook shows a practical end-to-end workflow:

- connect to the Slow Control MySQL database
- inspect tables and preview data
- read a time window
- align multiple tables onto a common timeline
- plot pressure-temperature (P-T) phase diagram paths


## Preparation

1. Copy `sc_config.example.json` to `sc_config.json` and fill in the database credentials.
2. Or set environment variables like `MYSQL_HOST`, `MYSQL_USER`, `MYSQL_PASSWORD`, `MYSQL_DATABASE`.
3. Make sure required packages are installed (see `pyproject.toml`).


In [ ]:
%load_ext autoreload
%autoreload 2

# Imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sc_reader import SCReader, MySQLConfig
from sc_reader import visualizer as viz

# Matplotlib settings
plt.rcParams["axes.unicode_minus"] = False  # show minus sign correctly

print("Imports OK")


## Connect to the database

`SCReader` will load connection settings from `sc_config.json` if it exists.
If not, it falls back to environment variables or default values.

Tip: keep credentials out of notebooks if you plan to share them.


In [ ]:
# Create reader (auto-loads sc_config.json / ~/.sc_config.json / env vars / defaults)
config = MySQLConfig.from_json("sc_config.example.json")
reader = SCReader(config=config)

print(f"Connected config: host={config.host}, db={config.database}")


In [ ]:
# List tables
reader.list_tables()

In [ ]:
%config InlineBackend.figure_format = "retina"  # for high-res plots 

In [ ]:
# Table schema / columns
reader.get_table_info("tempdata")

In [ ]:
# Preview data
reader.preview_table_data("piddata")

## Read data in a time range

Use ISO-like strings such as `YYYY-MM-DD HH:MM`. The time zone is controlled by
`SCReader(time_zone=...)`.


In [ ]:
start_time = "2026-04-01 00:00"
end_time = "2026-04-09"

# Full-range read
df_pid = reader.query_df("piddata", start_time, end_time)
df_runli = reader.query_df("runlidata", start_time, end_time)

In [ ]:
reader.query_df("tempdata", start_time, end_time)

In [ ]:
# Quick look
print(df_runli.head())


In [ ]:
# Plot pressure time series


pressure_cols = [c for c in df_runli.columns if c.lower().startswith("pressure")]

viz.plot_timeseries(df_runli, column=pressure_cols[:3], title="Pressure (all in one)")

In [ ]:
from pathlib import Path
from sc_reader import SCReader, MySQLConfig, TableSpec, AlignedData

cfg = MySQLConfig.from_json("/home/wxy/SC_Manager/sc_config.example.json")

cache_file = "/home/wxy/SC_Manager/data/parquet/aligned_piddata_cache.parquet"
wm_file = "/home/wxy/SC_Manager/data/parquet/aligned_piddata_watermark.json"

r = SCReader(config=cfg, state_path=wm_file)
specs = [
    TableSpec("tempdata", "timestamp"),
    TableSpec("runlidata", "timestamp"),
    TableSpec("statedata", "timestamp"),
    TableSpec("piddata", "timestamp"),
]

cache = AlignedData(
    r,
    specs,
    anchor="piddata",
    lookback="10s",
    tolerance="10s",
    cache_path=cache_file,
    auto_load=True,
    auto_save=False,
)

# 仅首次全量；后续都增量
cache.update(force_full=not Path(cache_file).exists())
cache.save(cache_file)

print("shape:", cache.data.shape)
print("columns sample:", list(cache.data.columns)[:30])
r.close()

In [ ]:
# Sampling interval summary (seconds)
print(df_runli.index.to_series().diff().dropna().describe())


## Time alignment (AlignedData)

AlignedData aligns multiple tables onto an anchor table's timeline.

- anchor: table whose timestamps define the output index
- tolerance: maximum allowed time difference when matching rows
- lookback: extra time window to handle late-arriving data

Aligned columns are named like `table__column`.


In [ ]:
# Aligned data uses column names like "table__column"
cache.data.head()


In [ ]:
# Slice a window and plot pressure again
cache.update()
start_time = "2026-04-01 00:00"
end_time = "2026-04-09"

df_all = cache[start_time:end_time]
mask = df_all.columns.str.contains(r"(?:press|pressure)", case=False, regex=True)
matched_cols = df_all.columns[mask].to_list()

viz.plot_timeseries(df_all, column=matched_cols, title="Pressure (all in one)")


## Plot P-T phase diagram paths

Make sure pressure is in bar and temperature is in Kelvin.
If your temperature is in Celsius, add 273.15 before plotting.
The example below plots three temperature channels against a single pressure channel.


In [ ]:
cache.columns

In [ ]:
from sc_reader import interactive_plot_pt_path

cache.update()
# Select time window for phase diagram
df_phase = cache["2026-4-17 13:10":"2026-04-17"]

# 1. 统一在这里配置要绘制的线条：每一行代表一条线，包含其专属的压力列、温度列、标签和颜色
plot_config = [
    {"label": "cold finger", "P_col": "runlidata__Pressure5", "T_col": "piddata__A_Temperature", "color": "red"},
    {"label": "Point B", "P_col": "runlidata__Pressure6", "T_col": "piddata__B_Temperature", "color": "orange"},
    {"label": "Point C", "P_col": "runlidata__Pressure6", "T_col": "piddata__C_Temperature", "color": "blue"},
    {"label": "Point D", "P_col": "runlidata__Pressure6", "T_col": "piddata__D_Temperature", "color": "green"},
    {"label": "Point E", "P_col": "runlidata__Pressure6", "T_col": "piddata__E_Temperature", "color": "black"},
    # {"label": "Pt 7", "P_col": "runlidata__Pressure6", "T_col": "tempdata__Temperature7", "color": "purple"},
    # {"label": "Pt 8", "P_col": "runlidata__Pressure6", "T_col": "tempdata__Temperature8", "color": "black"},
]

# 2. 自动检查所有需要的列是否存在
required_cols = list(set([cfg["P_col"] for cfg in plot_config] + [cfg["T_col"] for cfg in plot_config]))
missing = [c for c in required_cols if c not in df_phase.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}")


# 3. 使用时间子集包装一个临时 cache，避免一次性渲染整份数据
class TempCache:
    def __init__(self, data):
        self.data = data

    @property
    def columns(self):
        return self.data.columns

    def __getitem__(self, key):
        return self.data[key]


temp_cache = TempCache(data=df_phase)

# If temperatures are in Celsius, use temp_offset=273.15 below

# 4. 显示带时间轴联动的交互式相图
interactive_plot_pt_path(
    temp_cache,
    plot_config,
    gas="argon",
    arrow_max=12,
    arrow_min_dist=0.02,
    downsample_max_points=50000,
    T_range=(70, 110),
    P_range=(0.1, 3.5),
    temp_offset=0.0,
    width=900,
)


In [ ]:
import numpy as np

from sc_reader.phase_diagram import psat_bar


C_LOG4 = np.array([
    -1.696051349830958e-07,
    7.566233576453607e-05,
    -1.335292890301619e-02,
    1.158472642514562e00,
    -3.984494194563106e01,
])


def psat_argon_poly_80_100(T_K):
    """
    Argon psat (bar), polynomial fit valid for 80 <= T_K <= 100 K.
    Fit model: ln(Pbar) = poly4(T); Pbar = exp(lnP)
    """
    T = np.asarray(T_K, dtype=float)
    lnP = np.polyval(C_LOG4, T)
    return np.exp(lnP)


t_range = np.arange(80, 100, 1)
psat_argon_poly_80_100(t_range) - psat_bar(t_range)

In [ ]:
from sc_reader import psat_bar

T_RANGE = np.arange(80, 100, 1)
pressure_sat = psat_bar(T_RANGE)  # example usage

In [ ]:
pt = np.stack((T_RANGE, pressure_sat)).T